In [2]:
import polars as pl

df = pl.read_parquet("fandom-youtube-articles.parquet")

In [9]:
def all_categories_under_parent(df: pl.DataFrame, category: str) -> list[str]:
    p = df.filter(
        pl.col("namespace") == 14,
        pl.col("title").str.split(":").list.get(0) == "Category",
        pl.col("categories").list.contains(category)
    )
    if p.is_empty():
        return []
    inital = p.select(pl.col("title").str.split(":").list.get(1)).to_series().to_list()
    cats = set()
    for i in inital:
        cats.add(i)
        cats.update(all_categories_under_parent(df, i))
    return list(cats)
categories = all_categories_under_parent(df, "YouTubers")

In [15]:
yts = df.filter(
    pl.col("namespace") == 0,
    pl.col("categories").list.set_intersection(categories).list.len() > 0
).select(pl.col("title"), pl.col("text"), pl.col("categories"))
yts

title,text,categories
str,str,list[str]
"""Galipoka""","""{{Icons|Comedy|Vlogger|America…","[""Vlogging YouTubers"", ""American YouTubers"", … ""Terminated YouTubers""]"
"""Blunty""","""{{Icons|Reviewer|Animator|Vlog…","[""Review YouTubers"", ""Animation YouTubers"", … ""Users who joined in 2006""]"
"""Smosh""","""{{Icons|Wiki|Featured|HallofFa…","[""Comedy YouTubers"", ""Music YouTubers"", … ""Ten billion views""]"
"""BlameSociety""","""{{Stub}} {{Icons|Vlogger|Revie…","[""Users who joined in 2006"", ""American YouTubers"", … ""Review YouTubers""]"
"""Lonelygirl15""","""{{Icons|Wiki|Vlogger|American|…","[""Vlogging YouTubers"", ""American YouTubers"", … ""Terminated YouTubers""]"
…,…,…
"""Miriana Conte""","""{{Icons|Wiki|Musician|Maltese|…","[""Music YouTubers"", ""Maltese YouTubers"", … ""LGBT YouTubers""]"
"""Mortis""","""{{Icons|Commentary|Gaming|Musi…","[""Commentary YouTubers"", ""Gaming YouTubers"", … ""Users who joined in 2024""]"
"""マインクラフト 日本公式 / Minecraft Japan""","""#REDIRECT [[Minecraft]] [[Cat…","[""Gaming YouTubers"", ""Japanese YouTubers"", ""Users who joined in 2010""]"


In [35]:
import wikitextparser as wtp

def analyze_wikitext(text: str):
    parsed = wtp.parse(text)
    print(parsed)

yts.map_rows(
    lambda rows: analyze_wikitext(rows[1])
)


IOPub data rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_data_rate_limit`.

Current values:
ServerApp.iopub_data_rate_limit=1000000.0 (bytes/sec)
ServerApp.rate_limit_window=3.0 (secs)



{{Icons|Commentary|Reviewer|American|2009|YouTuber}}
{{Stub}}
{{NeedsCitations}}
{{YouTuber1
|title = ShadeX
|username = UCzViJ1z5gXvTWD9ac-34qwg
|image = ShadeX.jpg
|style = Commentary and Reviews
|join date = August 23, 2009
|Twitter = ShadeX98
|Instagram = shadex96
|other media = [https://www.twitch.tv/shadex98 Twitch]<br>[https://discord.gg/WgSHRZdznP Discord server]<br>[https://shadex98.tumblr.com/ Tumblr]<br>[https://www.tiktok.com/@shadex98 TikTok]
|vids = 191+
|update = Unscheduled
|status = Active
|nationality = American
|location = United States
|channel trailer = [[File:ShadeX Channel Intro|thumb|225 px]]
|most viewed video = [[File:Who is Ace? The Mysterious New Gorillaz Member (@ShadeX)|thumb|225 px]]
|first video = [[File:Mickey Kills Pikachu- SPOOF|thumb|225 px]]
}}
'''ShadeX''' {{BIRTHYEAR|1996}}, is an American [[YouTuber]] known for his commentary and review [[video]]s.  He typically talks about animated shows but sometimes talks about live action films along with mus

KeyboardInterrupt: 